# MSADS RAG Backend on Colab

Run the FastAPI backend on Colab and expose it via a Cloudflare quick-tunnel,
so a **locally-running frontend** can connect to it for demos when local hardware
cannot host `qwen3:8b`.

**Recommended runtime:** GPU (T4) — *Runtime → Change runtime type → T4 GPU*. Without GPU, qwen3:8b inference falls back to CPU and a single chat turn may take minutes.

**Session limits (free tier):** auto-disconnects after ~90 min idle or 12 h total. When the session dies the public URL also dies — re-run all cells to restore.

---
Run the cells **in order**. The first run takes ~5–10 min (mostly downloading `qwen3:8b`).

## 1. Install Ollama, cloudflared, and clone the repo

In [ ]:
# Install Ollama (LLM runtime). The current installer needs `zstd` for
# extraction; Colab's base image doesn't ship it, so install it first.
# All steps below are idempotent — safe to re-run.
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# Install cloudflared (public tunnel) — overwrite if already there.
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# Clone or update the project. We track the `new` branch where the
# streaming + Colab-demo work lives; flip BRANCH back to 'main' once it
# has been merged.
import os
REPO_DIR = '/content/RAG-based-Interactive-AI-for-MSADS-Website'
REPO_URL = 'https://github.com/Yoki-SyZhang/RAG-based-Interactive-AI-for-MSADS-Website.git'
BRANCH = 'new'

if not os.path.exists(os.path.join(REPO_DIR, 'app.py')):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch --quiet origin {BRANCH}
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull --ff-only origin {BRANCH}

%cd {REPO_DIR}

# Python deps
!pip install -q -r requirements.txt

## 2. Start Ollama and pull `qwen3:8b` (~5.2 GB, slow on first run)

In [22]:
import subprocess, time, urllib.request

# Idempotent re-run: kill any prior Ollama daemon before launching a new one.
subprocess.run(['pkill', '-f', 'ollama serve'], check=False)
time.sleep(2)

subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait until /api/tags responds
for _ in range(30):
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=2)
        print('Ollama is up.')
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('Ollama did not start within 30s')

# Pull the model — re-runs are no-ops if the weights are already cached.
!ollama pull qwen3:8b
!ollama list

Ollama is up.

NAME        ID              SIZE      MODIFIED               
qwen3:8b    500a1f067a9f    5.2 GB    Less than a second ago    


## 3. Start FastAPI and open a public tunnel

This cell launches `uvicorn` in the background, then starts a Cloudflare quick-tunnel and prints the public URL. Keep this cell's output open — closing it terminates the tunnel.

In [ ]:
import subprocess, time, threading, re, sys, urllib.request

# Idempotent re-run: kill any prior copy of uvicorn / cloudflared.
subprocess.run(['pkill', '-f', 'uvicorn'], check=False)
subprocess.run(['pkill', '-f', 'cloudflared'], check=False)
time.sleep(2)

uv_proc = subprocess.Popen(
    ['uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)

def _pump(proc, tag):
    for raw in iter(proc.stdout.readline, b''):
        sys.stdout.write(f'[{tag}] {raw.decode(errors="ignore")}')
        sys.stdout.flush()

threading.Thread(target=_pump, args=(uv_proc, 'uvicorn'), daemon=True).start()

# Wait for the FastAPI app to finish startup (lifespan loads the index)
for _ in range(60):
    try:
        urllib.request.urlopen('http://localhost:8000/health', timeout=2)
        print('FastAPI is up.')
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('FastAPI did not become ready within 60s')

# Start the Cloudflare quick-tunnel. trycloudflare.com sometimes fails to
# return a tunnel on the first try ("error code 1101") — retry up to 3 times.
public_url = None
for attempt in range(3):
    cf_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    for raw in iter(cf_proc.stdout.readline, b''):
        line = raw.decode(errors='ignore')
        sys.stdout.write(f'[cloudflared] {line}')
        sys.stdout.flush()
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            public_url = m.group(0)
            threading.Thread(target=_pump, args=(cf_proc, 'cloudflared'), daemon=True).start()
            break
        if 'failed to' in line.lower() or 'error code: 1101' in line.lower():
            cf_proc.terminate()
            break
    if public_url:
        break
    print(f'[cloudflared] tunnel attempt {attempt+1} failed; retrying in 3s...')
    time.sleep(3)

print('\n' + '=' * 70)
print(f'  Public backend URL : {public_url}')
print('=' * 70)
if public_url:
    print('\nLocal frontend command (run on your laptop, in the repo root):')
    print(f'  cd Frontend && VITE_BACKEND_URL={public_url} npm run dev')
    print('\nOpen http://localhost:5173 — Vite proxies /api/* to the Colab backend.')
else:
    print('\nTunnel did not come up. Re-run this cell, or check:')
    print('  https://www.cloudflarestatus.com/')

[cloudflared] 2026-05-10T02:00:11Z INF Initiating graceful shutdown due to signal terminated ...
[uvicorn] INFO:     Shutting down
[cloudflared] 2026-05-10T02:00:11Z ERR failed to run the datagram handler error="context canceled" connIndex=0 event=0 ip=198.41.192.27
[cloudflared] 2026-05-10T02:00:11Z ERR failed to serve tunnel connection error="accept stream listener encountered a failure while serving" connIndex=0 event=0 ip=198.41.192.27
[cloudflared] 2026-05-10T02:00:11Z ERR Serve tunnel error error="accept stream listener encountered a failure while serving" connIndex=0 event=0 ip=198.41.192.27
[cloudflared] 2026-05-10T02:00:11Z INF Retrying connection in up to 1s connIndex=0 event=0 ip=198.41.192.27
[cloudflared] 2026-05-10T02:00:11Z ERR Connection terminated connIndex=0
[cloudflared] 2026-05-10T02:00:11Z ERR no more connections active and exiting
[cloudflared] 2026-05-10T02:00:11Z INF Tunnel server stopped
[cloudflared] 2026-05-10T02:00:11Z INF Metrics server stopped
[uvicorn] IN

[cloudflared] 2026-05-10T02:00:21Z INF Generated Connector ID: 672f41d9-65f2-4bae-aad8-877dd1b6bbc2
[cloudflared] 2026-05-10T02:00:21Z INF Initial protocol quic
[cloudflared] 2026-05-10T02:00:21Z INF ICMP proxy will use 172.28.0.12 as source for IPv4
[cloudflared] 2026-05-10T02:00:21Z INF ICMP proxy will use ::1 in zone lo as source for IPv6
[cloudflared] 2026-05-10T02:00:21Z INF ICMP proxy will use 172.28.0.12 as source for IPv4
[cloudflared] 2026-05-10T02:00:21Z INF ICMP proxy will use ::1 in zone lo as source for IPv6
[cloudflared] 2026-05-10T02:00:21Z INF Starting metrics server on 127.0.0.1:20241/metrics
[cloudflared] 2026-05-10T02:00:21Z INF Tunnel connection curve preferences: [X25519MLKEM768 CurveP256] connIndex=0 event=0 ip=198.41.192.7
[cloudflared] 2026/05/10 02:00:21 failed to sufficiently increase receive buffer size (was: 208 kiB, wanted: 7168 kiB, got: 416 kiB). See https://github.com/quic-go/quic-go/wiki/UDP-Buffer-Sizes for details.
[cloudflared] 2026-05-10T02:00:21Z I

## 4. (Optional) Smoke test from inside Colab

Verify the backend answers before you switch the frontend over.

In [ ]:
import json, urllib.request

req = urllib.request.Request(
    'http://localhost:8000/chat',
    data=json.dumps({'query': 'How long is the program?', 'history': []}).encode(),
    headers={'Content-Type': 'application/json'},
)
with urllib.request.urlopen(req, timeout=300) as resp:
    payload = json.loads(resp.read())

print('Answer:\n', payload['answer'])
print('\nCitations:', len(payload['citations']))
for c in payload['citations']:
    print(f"  [{c['index']}] {c['title']} — {c['source_url']}")

In [ ]:
import subprocess, time, urllib.request

# 杀掉残留的 ollama 进程
subprocess.run(['pkill', '-f', 'ollama'], check=False)
time.sleep(2)

# 重启 daemon
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 等就绪
for _ in range(30):
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=2)
        print('Ollama is up again.')
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('Ollama did not start')

# 确认模型还在
!ollama list

In [ ]:
import subprocess, time, threading, sys, urllib.request

  # 先杀掉可能存在的僵尸 uvicorn
subprocess.run(['pkill', '-f', 'uvicorn'], check=False)
time.sleep(2)

uv_proc = subprocess.Popen(
    ['uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    cwd='/content/RAG-based-Interactive-AI-for-MSADS-Website',
)

def _pump(p, tag):
    for raw in iter(p.stdout.readline, b''):
        sys.stdout.write(f'[{tag}] {raw.decode(errors="ignore")}')
        sys.stdout.flush()
threading.Thread(target=_pump, args=(uv_proc, 'uvicorn'), daemon=True).start()

# 等就绪
for _ in range(60):
    try:
        urllib.request.urlopen('http://localhost:8000/health', timeout=2)
        print('FastAPI is up again.')
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('uvicorn did not come back')

In [ ]:
import subprocess, threading, sys, re

subprocess.run(['pkill', '-f', 'cloudflared'], check=False)

cf_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)

public_url = None
for raw in iter(cf_proc.stdout.readline, b''):
    line = raw.decode(errors='ignore')
    sys.stdout.write(f'[cloudflared] {line}')
    sys.stdout.flush()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group(0)
        def _pump(p):
            for r in iter(p.stdout.readline, b''):
                sys.stdout.write(f'[cloudflared] {r.decode(errors="ignore")}')
                sys.stdout.flush()
        threading.Thread(target=_pump, args=(cf_proc,), daemon=True).start()
        break

print(f'\nNew public URL: {public_url}')
print(f'\nRestart frontend with:\n  cd Frontend && VITE_BACKEND_URL={public_url} npm run dev')